<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/2_word_representation/Tutorial_2_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial Session 2: Tokenization

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.moma.org%2Fd%2Fassets%2FW1siZiIsIjIwMjMvMDYvMjcvMTcweHg5ZHcxZ19JTjI0MTdfMDA1X0NDQ1JfRnVsbF9zaXplLmpwZyJdLFsicCIsImNvbnZlcnQiLCItcXVhbGl0eSA5MCAtcmVzaXplIDIwMDB4NjY3XiAtZ3Jhdml0eSBDZW50ZXIgLWNyb3AgMjAwMHg2NjcrMCswIl1d%2FIN2417_005_CCCR-Full-size.jpg%3Fsha%3Dfae4b38a360c0c32&f=1&nofb=1&ipt=f2ccb3b3225eaa9f848cb952197286fdfab98d8cefdc02878eac556ae0158170&ipo=images$0)
<p align="right">
  <i>Rhapsody (detail)</i>. Jennifer Bartlett. (1975–76). Enamel on steel. <br>© 2023 Jennifer Bartlett. Photograph: John Wronn
</p>




This notebook contains three section:
- [Tokenizers exploration](#exploration): use different 'off-the-shelf' tokenizers, play with the inputs and compare the outputs
- [BPE Implementation](#bpe): Implement you own version of the Byte-Pair Encoding algorithm and train you own tokenizer!
- [Zipf's Law](#zipf): Compute Zipf's law on real-world data (based on the tokenizer(s) of your choice, or even the one you trained!)

You can start by any of these sections.

<a name="exploration"></a>
# Tokenizers Exploration

Explore the tokenization results of different off-the-shelf tokenizers. By default (but feel free to test other methods), the following code includes tokenizers from:
- [`nltk`](https://www.nltk.org/)
- [`spaCy`](https://spacy.io/)
- [`tiktoken`](https://github.com/openai/tiktoken)(note: they also have an app if you want to directly [test it in browser](https://tiktokenizer.vercel.app/))

In [ ]:
# nltk
from nltk.tokenize import TreebankWordTokenizer
nltk_tokenizer = TreebankWordTokenizer()

# spacy
from spacy.lang.en import English
nlp = English()
spacy_tokenizer = nlp.tokenizer

# tiktoken
import tiktoken
tiktoken_tokenizer = tiktoken.get_encoding("gpt2") # we use the gpt-2 variant here, you can choose another model

In [ ]:
# build a dictionnary of tokenization methods to ease comparison

dict_tokenize = {
    "nltk.TreebankWordTokenizer": lambda text: nltk_tokenizer.tokenize(text),
    "spacy.English.tokenizer": lambda text: [tok for tok in spacy_tokenizer(text)], # decompose spacy output format
    "spacy.English.tokenizer_ids": lambda text: [(tok.orth, tok.whitespace_) for tok in spacy_tokenizer(text)], # decompose spacy output format and access token IDs
    "tiktoken.gpt-2_ids": lambda text: tiktoken_tokenizer.encode(text), # return integers IDs
    "tiktoken.gpt-2_tokens": lambda text: [
        tiktoken_tokenizer.decode([tok_id]) # decode to see the tokens
        for tok_id in tiktoken_tokenizer.encode(text) # return integers IDs
    ],
}

Here are a few sentences, play with your examples (different domains, language, alphabets; spelling errors, slang, neologism, inventions, etc.).

In [ ]:
sentences = [ # a few toy examples to test tokenization, feel free to add your own examples!
    "Times flies like an arrow; fruit flies like banana.",
    "Jean-Edouard n'aime pas l'IA.",
    "欲速则不达",
    "This cat is enooooormous.",
    "Notebook is weird. Weird is notebook",
    "jqgoiber amojgorj gnon ogbjg ojbrgoùbe uerbùerb"
]

In [ ]:
# Encoding

for s in sentences:
  print("-"*30)
  print(f"Original sentence: {s}")
  print("Tokenization:")
  for k, v in dict_tokenize.items():
    print(f"\t- {k}: {v(s)}")

In [ ]:
# Input reconstruction Encoding + Decoding

dict_decode = {
    "spacy.English.tokenizer_ids": lambda input_ids: "".join([
        nlp.vocab.strings[tok_id] + tok_ws  # spacy actually works with a hash scheme for storing unknown words
        for tok_id, tok_ws in input_ids
    ]),
    "tiktoken.gpt-2_ids": lambda input_ids: tiktoken_tokenizer.decode(input_ids),
}

In [ ]:
# Reconstruction

for s in sentences:
  print("-"*30)
  print(f"Original sentence: {s}")
  print("Reconstructed input (input -> tokenize -> ids -> decode -> reconstructed):")
  for k, v in dict_decode.items():
    encoded_input = dict_tokenize[k](s)
    print(f"\t- {k}: {v(encoded_input)}")

- How does each tokenizer handle punctuation, accents, non-Latin scripts, ...?
- Count the total number of generated tokens per sentence across different libraries. Do you observe any patterns?
- Do the individual 'tokens' have semantic meaning? Why?
- Compare the number of tokens for different languages (including 'low-resource' languages), what do you observe (esp. when using BPE-like tokenization)? What are the potential implications (you can think in terms of computational cost and access to technology)?

Bonus:

- Compute the dependency tree for the famous sentence by Groucho Marx: "Times flies like an arrow; fruit flies like banana.": How does the model understand this sentence?

<a name="bpe"></a>
# BPE Implementation

Implement you own version of the BPE algorithm, you can check the details in the [original article](https://www.derczynski.com/papers/archive/BPE_Gage.pdf), or on [Wikipedia](https://en.wikipedia.org/wiki/Byte-pair_encoding).

If you do not want to completely start from scratch, a pre-filled [skeleton of a `BPE` class is included below](#bpe_skeleton). If you use this code, you will have to implement the missing parts of the methods:
- the BPE-training loop in `train`, used to learn the merges to perform BPE
- the `_encode_word` logic used once the model is trained to apply the merge rules on given sequence of bytes, and get the token ids.
- the `decode` method which decodes a sequence of tokens (integer IDs) into a natural string.

This process is decomposed below to test the implementations of each part separately using a simple toy example.

PS.: In case you want to check / get inspiration, a proposed complete implementation is included [at the end of this notebook](#bpe_solution).

## Step-by-step decomposition

Toy examples are provided to test the implementation for each of the main components of the BPE algorithù:
- [`encoding`](#toy_encoding)
- [`decoding`](#toy_decoding)
- one iteration of the [`train`](#toy_train) loop

<a name="toy_encoding"></a>

### Encoding

The toy environment to test the `encoding` component are provided below. For this part, we will need:

- a vocabulary (`dict{int:str}`) which maps integer IDs to bytes / characters
- the reversed vocabulary (`dict{str:int}`) which maps bytes to integer IDs
- the 'learned' merged rank (`dict{tuple:int}`): the rank in which to operate the merges, gives the 'priority' of the merge given a pair of tokens
- the 'learned' merges (`dict{tuple:int}`): gives the new token ID given a pair of tokens to be merged


And with that, a test sentence (`str`), from which we can derive the actual decomposition by hand (!), to make sure that it is working.

These variables are defined below:

In [ ]:
# Definition of the toy environment
# For the decoding function, we will need:

toy_vocab = {
  0: b' ',
  1: b'a',
  2: b'c',
  3: b'e',
  4: b'h',
  5: b'm',
  6: b'n',
  7: b'o',
  8: b's',
  9: b't',
  10: b'th',
  11: b'the',
  12: b'on' ,
  13: b'at',
  14: b'cat',
  15: b'thecat',
}
toy_token_to_id = {v:k for k, v in toy_vocab.items()}

toy_merge_ranks = {
    (9, 4): 0, # t, h -> th
    (10, 3): 1, # th, e -> the
    (7, 6): 2, # o, n -> on
    (1, 9): 3, # a, t -> at
    (2, 13): 4, # c, at -> cat
    (11, 14): 5, # the, cat -> the cat
}

toy_merge_tokens = {
    (9, 4): 10, # t, h -> th
    (10, 3): 11, # th, e -> the
    (7, 6): 12, # o, n -> on
    (1, 9): 13, # a, t -> at
    (2, 13): 14, # c, at -> cat
    (11, 14): 15, # the, cat -> thecat
}


test_sentence = "thecatsatonthemat"


# Process: Try to do it by hand!
# 'thecatsatonthemat'
# -> ['t', 'h', 'e', 'c', 'a', 't', 's', 'a', 't', 'o', 'n', 't', 'h', 'e', 'm', 'a', 't']
# -> ['th', 'e', 'c', 'a', 't', 's', 'a', 't', 'o', 'n', 'th', 'e', 'm', 'a', 't']
# -> ['the', 'c', 'a', 't', 's', 'a', 't', 'o', 'n', 'the', 'm', 'a', 't']
# -> ['the', 'c', 'a', 't', 's', 'a', 't', 'on', 'the', 'm', 'a', 't']
# -> ['the', 'c', 'at', 's', 'at', 'on', 'the', 'm', 'at']
# -> ['the', 'cat', 's', 'at', 'on', 'the', 'm', 'at']
# -> ['thecat', 's', 'at', 'on', 'the', 'm', 'at']
# -> [15, 8, 13, 12, 11, 5, 13]
# --------------------------
# Expected Output:
# [15, 8, 13, 12, 11, 5, 13]

In [ ]:
# TODO - IMPLEMENT THE FUNCTION

def _toy_encode_word(
    word:str,
    token_to_id:dict=toy_token_to_id,
    merge_rank:dict=toy_merge_ranks,
    merge_token:dict=toy_merge_tokens,
):
      """
      Given a string, encode it using the BPE logic,
      by iteratively merging tokens (with priorities)
      from the vocabulary / token_to_id
      """
      if not word:
          return []

      # start with the individual byte token IDs
      tokens = [token_to_id[bytes([b])] for b in word]

      # TODO : Implement the encoding loop
      while len(tokens) >= 2:
          # Step-by-step:
          # 1. Find all candidate pairs in the current token sequence (= all consecutive pairs)
          pairs = # TODO

          # 2. Get the merge rank for each pair
          # (with 'inf' if the pair was never learned)
          pairs_ranks = # TODO
          # Pair to merge is the one with the smaller rank
          pair_to_merge = # TODO

          # Stop if no remaining adjacent pairs
          if pair_to_merge not in merge_rank:
              break

          # 3. Perform the merge and update word tokenization
          new_tokens = [] # to be filled with the new tokens
          i = 0 # to iterate over the tokens chain

          # Retrieve merged token ID based on the 'learned' (given) dict: pair -> ID
          merged_id = merge_token[pair_to_merge]

          while i < len(tokens):
              if (
                  i < len(tokens) - 1
                  and (tokens[i], tokens[i + 1]) == pair_to_merge
              ): # merge pair!
                  # TODO: update new_tokens
              else: # continue iterating
                  # TODO: update new_tokens

          # update tokens decomposition with merge
          tokens = new_tokens

      return tokens


In [ ]:
# Once the function implemented, you can run this cell!
# Expected output: [15, 8, 13, 12, 11, 5, 13]

toy_token_ids = _toy_encode_word(test_sentence.encode("utf-8"))

print(toy_token_ids)

### Decoding

Then, to test the decoding part, we will simply re-use the variables we defined previously.

We will decode the output that we obtained (`[15, 8, 13, 12, 11, 5, 13]`), and we should find back the original `test_sentence`.

In [ ]:
# TODO - IMPLEMENT THIS FUNCTION

def toy_decode(
    ids:list,
    vocab:dict=toy_vocab,
  ):
    """
    Turn list of IDs into tokens and decode UTF-8.
    """

    raise NotImplementedError("You need to implement the `toy_decode` function!")

In [ ]:
# Once the function implemented, you can run this cell!

toy_decoded_input = toy_decode(toy_token_ids)

if toy_decoded_input == test_sentence:
  print(f"✅ Success! The decoded input is the same as the original input: {toy_decoded_input}")
else:
  print(f"❌ Oops it looks like something went wrong: the decoded string '{toy_decoded_input}' is not the same as the original input '{test_sentence}'.")

### BPE training loop

We have implemented the encoding and decoding schemes of a BPE algorithm. However, we started from a toy example with manually defined vocabulary, merges, priorities, etc... in practice all these variables are learned by the model!

The training process of the BPE algorithm is iterative. To make it simpler, let's start by implementing **one iteration** of the training loop. If everything works well, we can then just repeat the process until the vocabulary converges.

To test that it is working correctly, for which we can easily replicate the BPE process by hand. We will simply start from a base vocabulary (`dict{str:int}`) and its reverse ID to string mapping (`dict{int:str}`).

In [ ]:
from collections import Counter, defaultdict
import heapq

In [ ]:
# Definition of the toy environment
# to test one iteration of the training loop

toy_words = ["low", "lower", "lowest"]

toy_vocab = {0: b'l', 1: b'o', 2: b'w', 3: b'e', 4: b'r', 5: b's', 6: b't'} # only the characters in the toy corpus
toy_token_to_id = {v:k for k,v in toy_vocab.items()}


# Process: Try to do it by hand!
# Count pair(s):
#   ('l' - 'o', 3 times) ; ('o' - 'w', 3 times) ; ('w' - 'e', 2 times) ; ...
# Select most frequent: ('l' - 'o', 3 times)
# Create new token: ('l', 'o') -> 'lo' | New Id: len(vocab) = 7
# Then: update all affected words, and repercussions

First, we need to prepare for the training loop: convert the data into the right format and compute initial statistics.

In [ ]:
# Prepare the data before the training loop

# turn characters to bytes for each 'word'
data = [
    [bytes([b]) for b in word.encode("utf-8")]
    for word in toy_words
]

# Compute Initial Statistics
pair_count = Counter()
pair_words = defaultdict(set)
for word_id, word in enumerate(data): # for each 'word'
    for pair in zip(word, word[1:]):  # iterate over consecutive bytes
        pair_count[pair] += 1         # count byte pairs
        pair_words[pair].add(word_id) # list of words containing pair

# Priority queue (min-heap using negative counts) to rapidly query the most frequent pair
# This format is used for efficiency, but it could be conceptually avoided
heap = [
    (-count, pair)
    for pair, count in pair_count.items()
]
heapq.heapify(heap)

In [ ]:
print("Pair counts:")
print(pair_count)
print("Pair words:")
print(dict(pair_words))
print("Heap Queue:")
print(heap)

Now, we can implement the training iteration:

In [ ]:
# TODO - IMPLEMENT THIS FUNCTION

def one_bpe_iteration(
    data,
    vocab,
    token_to_id,
    pair_count,
    pair_words,
    heap,
):
    """
    Perform exactly ONE iteration of the BPE training loop.

    The function should:
        1. Find the most frequent pair.
        2. Create a new token from that pair.
        3. Add the new token to the vocabulary.
        4. Update all affected words.
        5. Update pair_count and pair_words.
        6. Update the heap.

    Return the updated data structures.
    """
    # Objects to update
    new_data = data.copy()
    new_vocab = vocab.copy()
    new_token_to_id = token_to_id.copy()
    new_pair_count = pair_count.copy()
    new_pair_words = pair_words.copy()
    new_heap = heap.copy()

    # 1. Find most frequent pair
    while heap:
      neg_count, pair = heapq.heappop(heap) # pop = get and remove from heap
      count = -neg_count
      if pair_count.get(pair, 0) == count: # count is outdated => discarded
        break # => go to training loop
    else:
      return (new_data, new_vocab, new_token_to_id, new_pair_count, new_pair_words, new_heap)
    if count < 2:
      return (new_data, new_vocab, new_token_to_id, new_pair_count, new_pair_words, new_heap)

    # 2. Create new token from that pair
    a, b = pair
    new_token = a+b
    new_id = len(vocab)

    print(
        f"{new_id:5d}: "
        f"{a!r} + {b!r} -> {new_token!r} "
        f"({count:,})"
    )

    # 3. Record new token in vocabulary
    new_vocab.update({new_id:new_token})
    new_token_to_id.update({new_token:new_id})

    # 4. Update all affected word
    affected = list(pair_words[pair])

    for word_id in affected:
      word = data[word_id]
      old_pairs = list(zip(word, word[1:]))

      # 5. Update pair_counts and pair_words & 6. the heap
      for p in old_pairs:
        new_pair_count[p] -= 1
        new_pair_words[p].discard(word_id)

      new_word = []
      i = 0
      while i < len(word):
        if (
          i + 1 < len(word)
          and word[i] == a
          and word[i + 1] == b
        ): # if we see the current pair
          new_word.append(new_token) # replace (a,b) -> new_token
          i += 2 # advance past both previous tokens
        else: # if not the current pair
          new_word.append(word[i]) # keep same token
          i += 1 # advance to next token

      new_data[word_id] = new_word

      new_pairs = list(zip(new_word, new_word[1:]))
      for p in new_pairs:
        new_pair_count[p] += 1
        new_pair_words[p].add(word_id)
        # push updated pair frequency back to priority queue
        heapq.heappush(
            new_heap,
            (-new_pair_count[p], p)
        )

    return (new_data, new_vocab, new_token_to_id, new_pair_count, new_pair_words, new_heap)

    # raise NotImplementedError("You need to implement the BPE training iteration!")

In [ ]:
# Once the function implemented, you can run this cell!
# Expected output: "7: b'l' + b'o' -> b'lo' (3)"

new_data, new_vocab, new_token_to_id, new_pair_count, new_pair_words, new_heap = one_bpe_iteration(
    data=data,
    vocab=toy_vocab,
    token_to_id=toy_token_to_id,
    pair_count=pair_count,
    pair_words=pair_words,
    heap=heap,
)

Make sure that the new token appears in the updated vocabulary:

In [ ]:
print(new_vocab)

In [ ]:
# Now you can duplicate this cell to keep expanding BPE vocabulary and learning merges!

new_data, new_vocab, new_token_to_id, new_pair_count, new_pair_words, new_heap = one_bpe_iteration(
    data=new_data,
    vocab=new_vocab,
    token_to_id=new_token_to_id,
    pair_count=new_pair_count,
    pair_words=new_pair_words,
    heap=new_heap,
)

<a name="bpe_skeleton"></a>

## `BPE` class implementation

From scratch, or using the methods implemented in the context of the toy settings provided above, complete the missing parts of this BPE class:

In [ ]:
from collections import Counter, defaultdict
import heapq
import regex


class BPE:
    def __init__(
        self,
        regex_pattern:str = r" ?\w+| ?\S+|\s+",
    ):
        """
        Initialise the BPE tokenizer with the 2**8 distinct byte values.

        Parameters:
            - regex_pattern [str | default: r" ?\w+| ?\S+|\s+"]
                Regular expression used to pre-tokenize text into categories
                to prevent BPE merges across natural boundary types.
                The default value can be decomposed as follow:
                    - ' ?\w+' matches optional space followed by letters/digit/underscores
                    - ' ?\S+' matches optional space followed by punctuations/symbols
                    - '\s+'   fallback to any remaining standalone whitesapce
                For reference, the regex used by gpt-2 tokenizer is: "'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

        """
        # ----------------- Vocabulary
        # Token ID -> bytes -- instantiated with the default 256 individual byte values
        self.vocab = {
            i: bytes([i])
            for i in range(256)
        }
        # Bytes -> token ID -- reverse 'self.vocab' dictionnary
        self.token_to_id = {
            token: i
            for i, token in self.vocab.items()
        }

        # ----------------- Merges
        # Complete list of learned merges (during training)
        self.merges = []
        # Merge rank (to speed up encoding, avoid full lookup)
        # (token_a, token_b) -> merge rank
        self.merge_rank = {}
        # (token_a, token_b) -> resulting token ID
        self.merge_token = {}

        # ----------------- Pre-tokenization rule
        self.regex_pattern = regex_pattern
        self._pat = regex.compile(regex_pattern)

    # ==========================
    # ~~~~~~~~ Training ~~~~~~~~
    # ==========================

    def train( # TODO: implement the missing parts
        self,
        corpus: str,
        vocab_size: int,
        verbose: bool = True
    ):
        """
        Train the BPE tokenizer on the given corpus:
            - learn byte-pair merges
            - expand vocabulary up to predetermined size
            - store merge priorities

        Parameters:
            - corpus [str]
                The textual corpus used to learn the merges.
            - vocab_size [int]
                The (maximal) size of the vocabulary.
                Note that vocab_size must be greater than 256 (number of individual bytes).
                The training algorithm will stop either
                    -> when the vocabulary is extended to vocab_size, or
                    -> when there are no more merges to do.
            - verbose [bool | default = True]
                Whether to print the iterative pair merges found during training or not.
        """

        if vocab_size < 256:
            raise ValueError(
                "vocab_size must be at least 256."
            )

        # ----------------- PRE-TRAINING SETUP -----------------
        # pre-tokenize the corpus based on regular expression into 'words'
        words = self._pat.findall(corpus)

        # turn characters to bytes for each 'word'
        data = [
            [bytes([b]) for b in word.encode("utf-8")]
            for word in words
        ]

        # Compute Initial Statistics
        pair_count = Counter()
        pair_words = defaultdict(set)
        for word_id, word in enumerate(data): # for each 'word'
            for pair in zip(word, word[1:]):  # iterate over consecutive bytes
                pair_count[pair] += 1         # count byte pairs
                pair_words[pair].add(word_id) # list of words containing pair

        # Priority queue (min-heap using negative counts) to rapidly query the most frequent pair
        # This format is used for efficiency, but it could be conceptually avoided
        heap = [
            (-count, pair)
            for pair, count in pair_count.items()
        ]
        heapq.heapify(heap)

        # -----------------  BPE TRAINING LOOP -----------------
        while len(self.vocab) < vocab_size: # iterate while reaching given vocab_size
            # TODO: Implement the training loop
            # 1. retrieve the current most frequent pair (a, b)
             # (be careful if complying with heapq, it has a negative count logic, check the documentation if needed)
             # 1'. Stop training if no more pairs to merge
            # 2. Define the new token (new_tok) and it's ID (new_id)
            # 3. Record the changes

            if verbose: # you should have all these variables now
                print(
                    f"{new_id}: "
                    f"{a!r} + {b!r} -> {new_token!r} "
                    f"({count:,})"
                )

            # record the merge rule and register the new token in the vocabulary (+ reverse)
            self.merges.append(pair)
            self.vocab[new_id] = new_token
            self.token_to_id[new_token] = new_id

            # retrieve only the words that contain the target pair (for efficiency: avoids scanning the full corpus)
            affected = list(pair_words[pair])

            # TODO: update the representations
            # For each affected word (which contains the new merged token)
            # 3.
            for word_id in affected:
                word = data[word_id]

                # 1. Decrement counts for all adjacent pairs in the un-merged version

                # 2. Replace every occurence of consecutive (a, b) with the new token
                new_word = []
                i = 0
                while i < len(word):
                    # TODO: implement step 2.

                # update corpus with the newly merged structure: (a,b) -> new_token
                data[word_id] = new_word

                # 3. Increment count for the new adjacent paris after the introduction of the (a,b)-merge
                # increment count for the new adjacent pairs after the introduction of the (a,b)-merge
                new_pairs = list(zip(new_word, new_word[1:]))
                for p in new_pairs:
                    # TODO: implement step 3
                    pass

        # ----------------- POST-TRAINING LOOKUP SETUP -----------------
        # build fast lookup tables for encoding text
        self.merge_rank.clear()
        self.merge_token.clear()

        for rank, (a, b) in enumerate(self.merges): # in the order of the learned merges
            # map byte tokens to their assigned integer vocabulary IDs
            a_id = self.token_to_id[a]
            b_id = self.token_to_id[b]

            new_id = self.token_to_id[a + b]

            # priority rank: lower rank means the merge was learned earlier (higher precedence)
            self.merge_rank[(a_id, b_id)] = rank
            # map candidate token ID pairs to the resulting merged token ID
            self.merge_token[(a_id, b_id)] = new_id

    # ==========================
    # ~~~~~~~~ Encoding ~~~~~~~~
    # ==========================

    def encode(
        self,
        text: str,
    ):
        """
        Encode raw text into a sequence of BPE token IDs

        Steps:
        1. Pre-tokenize input text into 'words' using the regex pattern.
        2. Encode each word into bytes and map them to subword token IDs. (TODO: IMPELMENT THE METHOD)
        3. Flatten and concatenate the resulting token IDs into a single list.

        Parameters:
            - text [str]
                The string to be tokenized.

        Returns:
            - list[int]
                List of integer token IDs representing the tokenized text.
        """
        output = []

        for word in self._pat.findall(text):
            output.extend(
                self._encode_word( # TODO: you need to implement this method
                    word.encode("utf-8")
                )
            )

        return output

    def _encode_word(self, data: bytes):
        """
        Encode a single sequence of bytes into token IDs by iteratively applying
        the learned merges in order of priority.
        """
        if not data:
            return []

        # start with the individual byte token IDs
        tokens = [self.token_to_id[bytes([b])] for b in data]

        while len(tokens) >= 2:
            # TODO: implement the encoding loop!
            pass

        return tokens


    # ==========================
    # ~~~~~~~~ Decoding ~~~~~~~~
    # ==========================

    def decode(self, ids:list):
        """
        Turn list of IDs into tokens and decode UTF-8.
        """

        # TODO: implement the decoding strategy (based on vocabulary lookup)

        return decoded_data

Once implemented, you can train your own BPE tokenizer on the corpora of your choice. Feel free to play with the parameters: how does it impact the results? What happens when increasing/decreasing `n_vocab`? What is the impact of the pre-tokenizing regular expression pattern on the BPE dynamics?

In [ ]:
%%time

# Example use case to test your implementation
# Note: took approx. 50 seconds on the 'War and Peace' book with n_vocab=50257

try:
  corpus = txt # your corpus here!
except:
  corpus = "The cat is on the mat. The cat is sleeping on the mat. The cat is on the sofa."


bpe = BPE()

text = "the cat sat on the mat"

print("Before Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

print("\nTraining the BPE model...\n")

bpe.train(
    corpus,
    vocab_size=50257, # same vocabulary size as tiktoken's gpt2 model
)

print("\nAfter Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

<a name="zipf"></a>
# Zipf's Law

1. Download book(s) from the [Project Guteberg](https://www.gutenberg.org/) website (preferrably in `.txt` format), for instance:
  - [the complete works of Shakespeare](https://www.gutenberg.org/ebooks/100)
  - [Tolstoy's *War and Peace*](https://www.gutenberg.org/ebooks/2600)
2. Tokenize the text with the method(s!) of you choice
3. Compute and visualise the $f(r)$ plot

- Can you find the Zipf's law? (Challenge: find the empirical coefficient of the [Zipf-Mandlebrot's Law](https://en.wikipedia.org/wiki/Zipf%27s_law$) for your data).
- Do you observe any differences when using different tokenizers?
- What are the most/least frequent tokens?
- You can also try with different languages, what do you observe?

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
# Quick drafty helpers
def make_zipf_plot(
    sorted_counts_dict:dict={},
    log_plot:bool=True
):
    """
    Make the Zipf's plot
    Inputs:
      - sorted_counts_dict [dict]
        Format: {<METHOD_NAME> [str] : <COUNTS> [list]}
        A dictionnary with
          - keys: identifiers of the methods
          - values: sorted list of token counts
      - log_plot [bool:True]
        Whether to use a log-log scale on the plot.
    """
    fig, ax = plt.subplots(figsize=(10,10))
    for k, v in sorted_counts_dict.items():
        ax.plot(
            np.arange(len(v))+1,
            v,
            label=k,
        )

    ax.set_xlabel("Rank")
    ax.set_ylabel("Frequency")
    ax.legend()

    if log_plot:
        plt.loglog()
    plt.show()

def cumulative_freq_plot(
    sorted_counts_dict:dict,
    normalise_rank:bool=False
):
    fig, ax = plt.subplots(figsize=(10,10))
    for k, v in sorted_counts_dict.items():
        cumulative_sum = v.cumsum()
        cumulative_freq = cumulative_sum/cumulative_sum[-1]
        ax.plot(
            np.arange(len(cumulative_freq))+1 if not normalise_rank else (np.arange(len(cumulative_freq))+1)/len(cumulative_freq),
            cumulative_freq,
            label=k,
        )

    ax.set_xlabel("Rank" if not normalise_rank else "Normalised Rank")
    ax.set_ylabel("Cumulative Frequency")
    ax.legend()
    plt.plot()

In [ ]:
# load your input data

file_path = "pg79657.txt" # Replace with the path to your file

with open(os.path.join(file_path)) as f: # for a .txt file
    txt = f.read()
f.close()

print(f"Loaded file {file_path}: len = {len(txt):,}.")

In [ ]:
# TODO: tokenize the text

tokens = txt.split() # TODO: replace with the method you chose

In [ ]:
# count individual tokens

counter = Counter(tokens)

sorted_counts = np.fromiter(counter.values(), dtype=int)
sorted_counts.sort()
sorted_counts = sorted_counts[::-1]

print(f"The twenty most common tokens are: {counter.most_common(20)}.")

In [ ]:
# prepare format to comply with the helper function to make the plot

frequencies_dict = {
    "my_tokenization": sorted_counts
}

In [ ]:
# plot f(r)

make_zipf_plot(
    frequencies_dict
)

<a name="bpe_solution"></a>

**Proposed BPE Solution**

In [ ]:
from collections import Counter, defaultdict
import heapq
import regex


class BPE:
    def __init__(
        self,
        regex_pattern:str = r" ?\w+| ?\S+|\s+",
    ):
        """
        Initialise the BPE tokenizer with the 2**8 distinct byte values.

        Parameters:
            - regex_pattern [str | default: r" ?\w+| ?\S+|\s+"]
                Regular expression used to pre-tokenize text into categories
                to prevent BPE merges across natural boundary types.
                The default value can be decomposed as follow:
                    - ' ?\w+' matches optional space followed by letters/digit/underscores
                    - ' ?\S+' matches optional space followed by punctuations/symbols
                    - '\s+'   fallback to any remaining standalone whitesapce
                For reference, the regex used by gpt-2 tokenizer is: "'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

        """
        # ----------------- Vocabulary
        # Token ID -> bytes -- instantiated with the default 256 individual byte values
        self.vocab = {
            i: bytes([i])
            for i in range(256)
        }
        # Bytes -> token ID -- reverse 'self.vocab' dictionnary
        self.token_to_id = {
            token: i
            for i, token in self.vocab.items()
        }

        # ----------------- Merges
        # Complete list of learned merges (during training)
        self.merges = []
        # Merge rank (to speed up encoding, avoid full lookup)
        # (token_a, token_b) -> merge rank
        self.merge_rank = {}
        # (token_a, token_b) -> resulting token ID
        self.merge_token = {}

        # ----------------- Pre-tokenization rule
        self.regex_pattern = regex_pattern
        self._pat = regex.compile(regex_pattern)

    # ==========================
    # ~~~~~~~~ Training ~~~~~~~~
    # ==========================

    def train(
        self,
        corpus: str,
        vocab_size: int,
        verbose: bool = True
    ):
        """
        Train the BPE tokenizer on the given corpus:
            - learn byte-pair merges
            - expand vocabulary up to predetermined size
            - store merge priorities

        Parameters:
            - corpus [str]
                The textual corpus used to learn the merges.
            - vocab_size [int]
                The (maximal) size of the vocabulary.
                Note that vocab_size must be greater than 256 (number of individual bytes).
                The training algorithm will stop either
                    -> when the vocabulary is extended to vocab_size, or
                    -> when there are no more merges to do.
            - verbose [bool | default = True]
                Whether to print the iterative pair merges found during training or not.
        """

        if vocab_size < 256:
            raise ValueError(
                "vocab_size must be at least 256."
            )

        # ----------------- PRE-TRAINING SETUP -----------------
        # pre-tokenize the corpus based on regular expression into 'words'
        words = self._pat.findall(corpus)

        # turn characters to bytes for each 'word'
        data = [
            [bytes([b]) for b in word.encode("utf-8")]
            for word in words
        ]

        # Compute Initial Statistics
        pair_count = Counter()
        pair_words = defaultdict(set)
        for word_id, word in enumerate(data): # for each 'word'
            for pair in zip(word, word[1:]):  # iterate over consecutive bytes
                pair_count[pair] += 1         # count byte pairs
                pair_words[pair].add(word_id) # list of words containing pair

        # Priority queue (min-heap using negative counts) to rapidly query the most frequent pair
        # This format is used for efficiency, but it could be conceptually avoided
        heap = [
            (-count, pair)
            for pair, count in pair_count.items()
        ]
        heapq.heapify(heap)

        # -----------------  BPE TRAINING LOOP -----------------
        while len(self.vocab) < vocab_size: # iterate while reaching given vocab_size
            # Pop pairs from heap until finding one whose count matches the current actual frequency
            while heap:
                neg_count, pair = heapq.heappop(heap) # pop = get and remove
                count = -neg_count
                if pair_count.get(pair, 0) == count:
                    break
            else: # Stop training: heap is empty = no more pairs available to merge
                break
            if count < 2: # Stop training if the most frequent pair appears less than twice
                break

            # unpack most frequent pair and create its merged byte representation
            a, b = pair
            new_token = a + b
            new_id = len(self.vocab)

            if verbose:
                print(
                    f"{new_id:5d}: "
                    f"{a!r} + {b!r} -> {new_token!r} "
                    f"({count:,})"
                )

            # record the merge rule and register the new token in the vocabulary (+ reverse)
            self.merges.append(pair)
            self.vocab[new_id] = new_token
            self.token_to_id[new_token] = new_id

            # retrieve only the words that contain the target pair (for efficiency: avoids scanning the full corpus)
            affected = list(pair_words[pair])

            for word_id in affected: # for each affected word (= which contains the new merged token)
                word = data[word_id]

                # decrement counts for all adjacent pairs in the current un-merged word
                old_pairs = list(zip(word, word[1:]))
                for p in old_pairs:
                    pair_count[p] -= 1
                    pair_words[p].discard(word_id)

                # replace every occurence of consecutive sequence (a, b) with the new token
                new_word = []
                i = 0
                while i < len(word):
                    if (
                        i + 1 < len(word)
                        and word[i] == a
                        and word[i + 1] == b
                    ): # if we see the current pair
                        new_word.append(new_token) # replace (a,b) -> new_token
                        i += 2 # advance past both previous tokens
                    else: # if not the current pair
                        new_word.append(word[i]) # keep same token
                        i += 1 # advance to next token

                # update corpus with the newly merged structure: (a,b) -> new_token
                data[word_id] = new_word

                # increment count for the new adjacent pairs after the introduction of the (a,b)-merge
                new_pairs = list(zip(new_word, new_word[1:]))
                for p in new_pairs:
                    pair_count[p] += 1
                    pair_words[p].add(word_id)
                    # push updated pair frequency back to priority queue
                    heapq.heappush(
                        heap,
                        (-pair_count[p], p)
                    )

        # ----------------- POST-TRAINING LOOKUP SETUP -----------------
        # build fast lookup tables for encoding text
        self.merge_rank.clear()
        self.merge_token.clear()

        for rank, (a, b) in enumerate(self.merges): # in the order of the learned merges
            # map byte tokens to their assigned integer vocabulary IDs
            a_id = self.token_to_id[a]
            b_id = self.token_to_id[b]

            new_id = self.token_to_id[a + b]

            # priority rank: lower rank means the merge was learned earlier (higher precedence)
            self.merge_rank[(a_id, b_id)] = rank
            # map candidate token ID pairs to the resulting merged token ID
            self.merge_token[(a_id, b_id)] = new_id

    # ==========================
    # ~~~~~~~~ Encoding ~~~~~~~~
    # ==========================

    def encode(
        self,
        text: str,
    ):
        """
        Encode raw text into a sequence of BPE token IDs

        Steps:
        1. Pre-tokenize input text into 'words' using the regex pattern.
        2. Encode each word into bytes and map them to subword token IDs.
        3. Flatten and concatenate the resulting token IDs into a single list.

        Parameters:
            - text [str]
                The string to be tokenized.

        Returns:
            - list[int]
                List of integer token IDs representing the tokenized text.
        """
        output = []

        for word in self._pat.findall(text):
            output.extend(
                self._encode_word(
                    word.encode("utf-8")
                )
            )

        return output

    def _encode_word(self, data: bytes):
        """
        Encode a single sequence of bytes into token IDs by iteratively applying
        the learned merges in order of priority.
        """
        if not data:
            return []

        # start with the individual byte token IDs
        tokens = [self.token_to_id[bytes([b])] for b in data]

        while len(tokens) >= 2:
            # find all candidate pairs in the current token sequence
            pairs = zip(tokens, tokens[1:])

            # get the merge rank for each pair (with None if the pair was never learned)
            # find the pair with smallest rank (highest priority merge)
            pair_to_merge = min(
                pairs,
                key = lambda p: self.merge_rank.get(p, float("inf"))
            )

            # stop if no remaining adjacent pairs
            if pair_to_merge not in self.merge_rank:
                break

            # otherwise: perform the merge and update word tokenization
            new_tokens = []
            i = 0
            merged_id = self.merge_token[pair_to_merge] # retrieve merged token id based from the highest priority pair

            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and (tokens[i], tokens[i + 1]) == pair_to_merge
                ): # merge pair!
                    new_tokens.append(merged_id)
                    i += 2  # skip past the merged pair
                else: # continue iterating
                    new_tokens.append(tokens[i])
                    i += 1

            # update tokens decomposition with merge
            tokens = new_tokens

        return tokens


    # ==========================
    # ~~~~~~~~ Decoding ~~~~~~~~
    # ==========================

    def decode(self, ids:list):
        """
        Turn list of IDs into tokens and decode UTF-8.
        """

        data = b"".join(
            self.vocab[i]
            for i in ids
        )

        return data.decode("utf-8")

In [ ]:
# Example use case

try:
  corpus = txt # your corpus here!
except:
  corpus = "The cat is on the mat. The cat is sleeping on the mat. The cat is on the sofa."


bpe = BPE()

text = "the cat sat on the mat"

print("Before Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

print("\nTraining the BPE model...\n")

bpe.train(
    corpus,
    vocab_size=50257, # same vocabulary size as tiktoken's gpt2 model
)

print("\nAfter Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])